In [ ]:
import pandas as pd

# Load user stats
users = pd.read_csv("data/processed/user_type_proportions.csv")
orders = pd.read_csv("data/processed/order_type_proportions.csv")

upc = "user_prop_ultra_processed_or_sugary_beverage_heavy"
fc  = "order_flagged_ultra_processed_or_sugary_beverage_heavy"

# Focus on persistent users: high user_prop, many orders
candidates = users[
    (users["n_orders"] >= 15) &
    (users[upc] >= 0.6)
][["user_id", "n_orders", upc]].sort_values("n_orders", ascending=False)

print(f"Persistent ultra-processed users (>=15 orders, user_prop>=0.6): {len(candidates):,}")
print(candidates.head(20).to_string(index=False))

In [1]:
# Pick a few strong candidates and look at their actual orders + products
TARGET_USERS = [33799, 71670, 12470, 32328]  # high prop, 60-82 orders

orders_all = pd.read_csv("data/processed/order_type_proportions.csv")
raw_orders = pd.read_csv("data/raw/orders.csv")

# Load order-product join
op_prior = pd.read_csv("data/raw/order_products__prior.csv")
op_train = pd.read_csv("data/raw/order_products__train.csv")
op = pd.concat([op_prior, op_train], ignore_index=True)

products = pd.read_csv("data/raw/products.csv")
flagged_prods = pd.read_csv("data/processed/products_flagged.csv")

# Merge product names + flag
prod_info = products.merge(
    flagged_prods[["product_id", "is_ultra_processed_or_sugary_beverage_heavy"]],
    on="product_id", how="left"
)

for uid in TARGET_USERS:
    user_orders = raw_orders[raw_orders["user_id"] == uid].sort_values("order_number")
    n_orders = len(user_orders)
    order_ids = user_orders["order_id"].tolist()

    # Get all items
    items = op[op["order_id"].isin(order_ids)].merge(prod_info, on="product_id", how="left")
    n_items_total = len(items)
    n_flagged = items["is_ultra_processed_or_sugary_beverage_heavy"].sum()

    # Most frequently bought flagged products
    top_flagged = (
        items[items["is_ultra_processed_or_sugary_beverage_heavy"] == True]
        .groupby("product_name").size()
        .sort_values(ascending=False)
        .head(10)
    )

    print(f"\n{'='*60}")
    print(f"User {uid} | {n_orders} orders | {n_items_total} total items | "
          f"{n_flagged} flagged ({100*n_flagged/n_items_total:.1f}%)")
    print(f"Order span: order #{user_orders['order_number'].min()} – #{user_orders['order_number'].max()}")
    print(f"\nTop ultra-processed products:")
    for name, cnt in top_flagged.items():
        print(f"  {cnt:3d}x  {name}")


User 33799 | 81 orders | 101 total items | 79 flagged (78.2%)
Order span: order #1 – #81

Top ultra-processed products:
   77x  Ice
    1x  Ranch Dressing
    1x  Squeeze Real Mayonnaise

User 71670 | 65 orders | 147 total items | 137 flagged (93.2%)
Order span: order #1 – #65

Top ultra-processed products:
   41x  Soda
   32x  Zero Calorie Cola
   23x  Cheez-It Baked Snack Crackers
   21x  Trail Mix
   12x  Organic Simply Naked Pita Chips
    4x  Wint-O-Green
    2x  Toasted Coconut Chips
    1x  Cereal
    1x  Organic Tortilla Chips

User 12470 | 72 orders | 328 total items | 281 flagged (85.7%)
Order span: order #1 – #72

Top ultra-processed products:
   35x  Multi Grain Cheerios
   32x  Cinnamon Toast Crunch
   24x  Chocolate Chip Cookies
   24x  Nutri Grain Bars Multi Pack
   20x  Cheez-It Baked Snack Crackers
   20x  Original Rice Krispies Treats
   19x  Raisin Bran
   19x  Cookie Bars
   19x  Chocolate Wafers Candy Bar
   18x  Frosted Flakes

User 32328 | 82 orders | 405 total 

In [2]:

# ── Survey snapshot: User 12470 ───────────────────────────────────────────────
SURVEY_USER = 12470

user_orders = raw_orders[raw_orders["user_id"] == SURVEY_USER].sort_values("order_number")
order_ids   = user_orders["order_id"].tolist()
items       = op[op["order_id"].isin(order_ids)].merge(prod_info, on="product_id", how="left")
items       = items.merge(user_orders[["order_id", "order_number", "order_dow",
                                        "order_hour_of_day", "days_since_prior_order"]],
                          on="order_id", how="left")
items["is_flagged"] = items["is_ultra_processed_or_sugary_beverage_heavy"].fillna(False)

DOW = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]

# ── Summary stats ─────────────────────────────────────────────────────────────
n_orders      = len(user_orders)
n_items       = len(items)
n_flagged     = items["is_flagged"].sum()
pct_flagged   = 100 * n_flagged / n_items
n_unique_prod = items["product_name"].nunique()
n_flagged_unique = items[items["is_flagged"]]["product_name"].nunique()
avg_items_per_order = n_items / n_orders
avg_days_between = user_orders["days_since_prior_order"].dropna().mean()

print("=" * 65)
print(f"  SURVEY SNAPSHOT — Anonymous User")
print("=" * 65)
print(f"  Total orders recorded          : {n_orders}")
print(f"  Total items purchased          : {n_items:,}")
print(f"  Avg items per order            : {avg_items_per_order:.1f}")
print(f"  Avg days between orders        : {avg_days_between:.1f}")
print(f"  Ultra-processed items          : {int(n_flagged):,} / {n_items:,}  ({pct_flagged:.1f}%)")
print(f"  Unique products ever bought    : {n_unique_prod}")
print(f"  Unique ultra-processed products: {n_flagged_unique}")

# ── Top ultra-processed products ──────────────────────────────────────────────
top = (items[items["is_flagged"]]
       .groupby("product_name").size()
       .sort_values(ascending=False)
       .head(15))

print(f"\n  Most frequently purchased ultra-processed items:")
for i, (name, cnt) in enumerate(top.items(), 1):
    bar = "█" * min(cnt, 40)
    print(f"  {i:2d}. {name:<42}  {cnt:3d}x  {bar}")

# ── Per-order timeline (last 15 orders) ───────────────────────────────────────
print(f"\n  Order-by-order timeline (most recent 15 of {n_orders}):")
print(f"  {'#':>3}  {'Day':<10}  {'Time':>5}  {'Items':>5}  {'% UP':>6}  Items purchased")
print("  " + "-" * 85)

recent_orders = user_orders.tail(15)
for _, row in recent_orders.iterrows():
    oid    = row["order_id"]
    onum   = int(row["order_number"])
    dow    = DOW[int(row["order_dow"])]
    hour   = int(row["order_hour_of_day"])
    time_s = f"{hour:02d}:00"

    order_items   = items[items["order_id"] == oid]
    n_oi          = len(order_items)
    n_oi_flag     = int(order_items["is_flagged"].sum())
    pct_oi        = 100 * n_oi_flag / n_oi if n_oi > 0 else 0

    # Show up to 4 item names, flagged ones marked with *
    def fmt(r):
        mark = "*" if r["is_flagged"] else " "
        return f"{mark}{r['product_name']}"
    sample_items = ", ".join(order_items.apply(fmt, axis=1).tolist()[:4])
    if n_oi > 4:
        sample_items += f", … (+{n_oi-4} more)"

    print(f"  {onum:3d}  {dow:<10}  {time_s:>5}  {n_oi:5d}  {pct_oi:5.0f}%  {sample_items}")

print(f"\n  * = ultra-processed/sugary item")
print(f"\n  What an algorithm infers from this basket history:")
print(f"    - Consistent high-sugar, high-processed-carb diet across {n_orders} orders")
print(f"    - {pct_flagged:.0f}% of all purchases are ultra-processed or sugary")
print(f"    - Pattern is stable over time (persistent regime)")
print(f"    - Dietary pattern of this type is associated with risk for")
print(f"      depression, anxiety, and cognitive decline in epidemiological literature")


  SURVEY SNAPSHOT — Anonymous User
  Total orders recorded          : 72
  Total items purchased          : 328
  Avg items per order            : 4.6
  Avg days between orders        : 4.5
  Ultra-processed items          : 281 / 328  (85.7%)
  Unique products ever bought    : 42
  Unique ultra-processed products: 25

  Most frequently purchased ultra-processed items:
   1. Multi Grain Cheerios                         35x  ███████████████████████████████████
   2. Cinnamon Toast Crunch                        32x  ████████████████████████████████
   3. Chocolate Chip Cookies                       24x  ████████████████████████
   4. Nutri Grain Bars Multi Pack                  24x  ████████████████████████
   5. Cheez-It Baked Snack Crackers                20x  ████████████████████
   6. Original Rice Krispies Treats                20x  ████████████████████
   7. Raisin Bran                                  19x  ███████████████████
   8. Cookie Bars                                  19x 

In [3]:
# ── Step 1: Find persistent alcohol users ─────────────────────────────────────
users_alc = pd.read_csv("data/processed/user_type_proportions.csv")

upc_alc = "user_prop_alcohol"

candidates_alc = users_alc[
    (users_alc["n_orders"] >= 15) &
    (users_alc[upc_alc] >= 0.6)
][["user_id", "n_orders", upc_alc]].sort_values("n_orders", ascending=False)

print(f"Persistent alcohol users (>=15 orders, user_prop>=0.6): {len(candidates_alc):,}")
print(candidates_alc.head(20).to_string(index=False))


Persistent alcohol users (>=15 orders, user_prop>=0.6): 741
 user_id  n_orders  user_prop_alcohol
    3377       100           0.840000
  159549       100           0.910000
  161200       100           0.660000
   81704       100           0.760000
   12772       100           0.810000
   93140       100           1.000000
  104339       100           0.620000
  109305        99           0.767677
  118609        99           0.979798
  142073        99           0.929293
   84478        99           0.949495
  120897        99           0.888889
  126714        99           0.666667
  191004        98           0.816327
   61220        94           0.829787
    5588        92           0.750000
  118297        91           0.626374
   98688        90           0.611111
   84715        89           0.842697
  111623        86           0.674419


In [4]:
# ── Step 2: Examine top alcohol candidates ────────────────────────────────────
# Rebuild prod_info with alcohol flag
prod_info_alc = products.merge(
    flagged_prods[["product_id", "is_alcohol"]],
    on="product_id", how="left"
)

TARGET_USERS_ALC = [93140, 118609, 84478, 142073, 12772]  # user_prop >= 0.81, 99-100 orders

for uid in TARGET_USERS_ALC:
    user_orders_u = raw_orders[raw_orders["user_id"] == uid].sort_values("order_number")
    n_orders_u = len(user_orders_u)
    order_ids_u = user_orders_u["order_id"].tolist()

    items_u = op[op["order_id"].isin(order_ids_u)].merge(prod_info_alc, on="product_id", how="left")
    n_items_u = len(items_u)
    n_flagged_u = items_u["is_alcohol"].sum()

    top_flagged_u = (
        items_u[items_u["is_alcohol"] == True]
        .groupby("product_name").size()
        .sort_values(ascending=False)
        .head(10)
    )

    print(f"\n{'='*60}")
    print(f"User {uid} | {n_orders_u} orders | {n_items_u} total items | "
          f"{n_flagged_u} flagged ({100*n_flagged_u/n_items_u:.1f}%)")
    print(f"Order span: order #{user_orders_u['order_number'].min()} – #{user_orders_u['order_number'].max()}")
    print(f"\nTop alcohol products:")
    for name, cnt in top_flagged_u.items():
        print(f"  {cnt:3d}x  {name}")



User 93140 | 100 orders | 373 total items | 191 flagged (51.2%)
Order span: order #1 – #100

Top alcohol products:
   70x  India Pale Ale
   26x  Extra IPA Beer
   26x  Vodka
   16x  Wooky Jack Black Rye Ipa
   13x  Brut California Champagne
   11x  Red Berry Vodka
   10x  Tricerahops Double IPA
    8x  Sculpin IPA
    6x  Whiskey
    1x  12 Year Old Single Malt Scotch Speyside

User 118609 | 99 orders | 587 total items | 120 flagged (20.4%)
Order span: order #1 – #99

Top alcohol products:
   72x  Tequila Blanco
   21x  Tequila Silver
    6x  Beer
    6x  Cabernet Sauvignon Wine
    5x  Tequila Reposado with Glass
    3x  Old Vine Zinfandel Wine
    2x  India Pale Ale
    2x  India Pale Ale Racer 5
    1x  California Red Wine
    1x  Premium Lager Beer

User 84478 | 100 orders | 378 total items | 94 flagged (24.9%)
Order span: order #1 – #100

Top alcohol products:
   94x  Vodka

User 142073 | 100 orders | 593 total items | 143 flagged (24.1%)
Order span: order #1 – #100

Top alcohol

In [5]:
# ── Step 3: Survey snapshot — Alcohol user 84478 ─────────────────────────────
SURVEY_USER_ALC = 84478

user_orders_alc = raw_orders[raw_orders["user_id"] == SURVEY_USER_ALC].sort_values("order_number")
order_ids_alc   = user_orders_alc["order_id"].tolist()
items_alc       = op[op["order_id"].isin(order_ids_alc)].merge(prod_info_alc, on="product_id", how="left")
items_alc       = items_alc.merge(
    user_orders_alc[["order_id", "order_number", "order_dow",
                     "order_hour_of_day", "days_since_prior_order"]],
    on="order_id", how="left"
)
items_alc["is_flagged"] = items_alc["is_alcohol"].fillna(False)

# ── Summary stats ─────────────────────────────────────────────────────────────
n_orders_a      = len(user_orders_alc)
n_items_a       = len(items_alc)
n_flagged_a     = items_alc["is_flagged"].sum()
pct_flagged_a   = 100 * n_flagged_a / n_items_a
n_unique_a      = items_alc["product_name"].nunique()
n_flagged_uniq_a = items_alc[items_alc["is_flagged"]]["product_name"].nunique()
avg_items_a     = n_items_a / n_orders_a
avg_days_a      = user_orders_alc["days_since_prior_order"].dropna().mean()

print("=" * 65)
print(f"  SURVEY SNAPSHOT — Anonymous User")
print("=" * 65)
print(f"  Total orders recorded          : {n_orders_a}")
print(f"  Total items purchased          : {n_items_a:,}")
print(f"  Avg items per order            : {avg_items_a:.1f}")
print(f"  Avg days between orders        : {avg_days_a:.1f}")
print(f"  Alcohol items                  : {int(n_flagged_a):,} / {n_items_a:,}  ({pct_flagged_a:.1f}%)")
print(f"  Unique products ever bought    : {n_unique_a}")
print(f"  Unique alcohol products        : {n_flagged_uniq_a}")

# ── Top alcohol products ──────────────────────────────────────────────────────
top_alc = (items_alc[items_alc["is_flagged"]]
           .groupby("product_name").size()
           .sort_values(ascending=False)
           .head(15))

print(f"\n  Most frequently purchased alcohol items:")
for i, (name, cnt) in enumerate(top_alc.items(), 1):
    bar = "█" * min(cnt, 40)
    print(f"  {i:2d}. {name:<42}  {cnt:3d}x  {bar}")

# ── Per-order timeline (last 15 orders) ───────────────────────────────────────
print(f"\n  Order-by-order timeline (most recent 15 of {n_orders_a}):")
print(f"  {'#':>3}  {'Day':<10}  {'Time':>5}  {'Items':>5}  {'% Alc':>6}  Items purchased")
print("  " + "-" * 85)

for _, row in user_orders_alc.tail(15).iterrows():
    oid    = row["order_id"]
    onum   = int(row["order_number"])
    dow    = DOW[int(row["order_dow"])]
    time_s = f"{int(row['order_hour_of_day']):02d}:00"

    order_items_a = items_alc[items_alc["order_id"] == oid]
    n_oi_a        = len(order_items_a)
    n_oi_flag_a   = int(order_items_a["is_flagged"].sum())
    pct_oi_a      = 100 * n_oi_flag_a / n_oi_a if n_oi_a > 0 else 0

    def fmt_alc(r):
        mark = "*" if r["is_flagged"] else " "
        return f"{mark}{r['product_name']}"
    sample_a = ", ".join(order_items_a.apply(fmt_alc, axis=1).tolist()[:4])
    if n_oi_a > 4:
        sample_a += f", … (+{n_oi_a-4} more)"

    print(f"  {onum:3d}  {dow:<10}  {time_s:>5}  {n_oi_a:5d}  {pct_oi_a:5.0f}%  {sample_a}")

print(f"\n  * = alcohol item")
print(f"\n  What an algorithm infers from this basket history:")
print(f"    - Alcohol present in {pct_flagged_a:.0f}% of all purchased items across {n_orders_a} orders")
print(f"    - Diverse product range: IPAs, vodka, champagne, whisky, scotch")
print(f"    - Pattern is stable and persistent over time (lift = 23× above chance)")
print(f"    - Regular alcohol purchasing patterns are associated with")
print(f"      alcohol use disorder risk, anxiety, and depression in clinical literature")


  SURVEY SNAPSHOT — Anonymous User
  Total orders recorded          : 100
  Total items purchased          : 378
  Avg items per order            : 3.8
  Avg days between orders        : 2.5
  Alcohol items                  : 94 / 378  (24.9%)
  Unique products ever bought    : 54
  Unique alcohol products        : 1

  Most frequently purchased alcohol items:
   1. Vodka                                        94x  ████████████████████████████████████████

  Order-by-order timeline (most recent 15 of 100):
    #  Day          Time  Items   % Alc  Items purchased
  -------------------------------------------------------------------------------------
   86  Saturday    08:00      4     25%  *Vodka,  1% Low Fat Milk,  Classic Soda,  Chicken Noodle Soup, Cup
   87  Tuesday     10:00      4     25%   1% Low Fat Milk, *Vodka,  Sparkling Natural Spring Water,  Classic Soda
   88  Thursday    11:00      2     50%  *Vodka,  1% Low Fat Milk
   89  Sunday      11:00      5     20%  *Vodka,  1% Lo

In [10]:
# ── Step 1: Find persistent GI/gluten-free users ─────────────────────────────
users_gf = pd.read_csv("data/processed/user_type_proportions.csv")
upc_gf = "user_prop_gi_sensitivity_gluten_free"

candidates_gf = users_gf[
    (users_gf["n_orders"] >= 15) &
    (users_gf[upc_gf] >= 0.6)
][["user_id", "n_orders", upc_gf]].sort_values("n_orders", ascending=False)

print(f"Persistent GI/gluten-free users (>=15 orders, user_prop>=0.6): {len(candidates_gf):,}")
print(candidates_gf.head(20).to_string(index=False))


Persistent GI/gluten-free users (>=15 orders, user_prop>=0.6): 681
 user_id  n_orders  user_prop_gi_sensitivity_gluten_free
   10747       100                              0.600000
   12886       100                              0.700000
   69995       100                              0.680000
  109020       100                              0.710000
  133227       100                              0.620000
   40278       100                              0.600000
     964        99                              0.828283
  101580        97                              0.876289
   88890        96                              0.635417
   97729        93                              0.698925
   84301        92                              0.804348
  170741        87                              0.666667
  179752        85                              0.729412
  204854        84                              0.797619
  203399        82                              0.646341
    1378        80   

In [11]:
# ── Step 2: Examine top GI/gluten-free candidates ────────────────────────────
prod_info_gf = products.merge(
    flagged_prods[["product_id", "is_gi_sensitivity_gluten_free"]],
    on="product_id", how="left"
)

# Focus on high user_prop with many orders: 964, 101580, 84301, 109020, 12886
TARGET_USERS_GF = [964, 101580, 84301, 109020, 12886]

for uid in TARGET_USERS_GF:
    user_orders_u = raw_orders[raw_orders["user_id"] == uid].sort_values("order_number")
    order_ids_u   = user_orders_u["order_id"].tolist()
    items_u       = op[op["order_id"].isin(order_ids_u)].merge(prod_info_gf, on="product_id", how="left")
    n_items_u     = len(items_u)
    n_flagged_u   = items_u["is_gi_sensitivity_gluten_free"].sum()

    top_flagged_u = (
        items_u[items_u["is_gi_sensitivity_gluten_free"] == True]
        .groupby("product_name").nunique()  # use nunique to just show distinct products
        .reindex(
            items_u[items_u["is_gi_sensitivity_gluten_free"] == True]
            .groupby("product_name").size()
            .sort_values(ascending=False)
            .index
        )
    )
    top_flagged_u = (
        items_u[items_u["is_gi_sensitivity_gluten_free"] == True]
        .groupby("product_name").size()
        .sort_values(ascending=False)
        .head(12)
    )
    n_unique_gf = items_u[items_u["is_gi_sensitivity_gluten_free"] == True]["product_name"].nunique()

    print(f"\n{'='*60}")
    print(f"User {uid} | {len(user_orders_u)} orders | {n_items_u} total items | "
          f"{n_flagged_u} flagged ({100*n_flagged_u/n_items_u:.1f}%) | {n_unique_gf} unique GF products")
    print(f"\nTop gluten-free/GI products:")
    for name, cnt in top_flagged_u.items():
        print(f"  {cnt:3d}x  {name}")



User 964 | 100 orders | 718 total items | 174 flagged (24.2%) | 31 unique GF products

Top gluten-free/GI products:
   27x  Original Gluten Free Crackers
   22x  Gluten Free Strawberry Toaster Pastry
   16x  Gluten Free Multigrain Bread
   12x  Gluten Free Original English Muffins
   12x  Gluten Free Rice Pasta & Cheddar Single Serving Macaroni & Cheese
   11x  Gluten Free Smoked Gouda Mac and Cheese
   10x  Gluten Free Whole Grain Bagels
    8x  Gluten Free Oatmeal Grahams Snack Packs 4 Ct
    8x  Gluten Free Uncured Frozen Pepperoni Pizza
    7x  Gluten Free Cinnamon Raisin Bread
    5x  Gluten Free Classic Hamburger Buns
    5x  Gluten Free Lemon Streusel Muffins

User 101580 | 98 orders | 1032 total items | 231 flagged (22.4%) | 34 unique GF products

Top gluten-free/GI products:
   65x  Gluten Free Brown Rice English Muffins 6 ct
   26x  Gluten Free Pizza Crust
   13x  Gluten Free Muffin Mix
   13x  Gluten Free Cinnamon Raisin Bread
   11x  Gluten Free Cheddar Macaroni & Cheese R

User 964 is the best pick — 31 unique GF products, diverse across crackers, pastries, bread, English muffins, pasta, pizza, bagels, cookies. It reads like a real celiac/GI-sensitive household, not a single-product habit. 

In [12]:
# ── Step 3: Survey snapshot — GI/Gluten-free user 964 ────────────────────────
SURVEY_USER_GF = 964

user_orders_gf = raw_orders[raw_orders["user_id"] == SURVEY_USER_GF].sort_values("order_number")
order_ids_gf   = user_orders_gf["order_id"].tolist()
items_gf       = op[op["order_id"].isin(order_ids_gf)].merge(prod_info_gf, on="product_id", how="left")
items_gf       = items_gf.merge(
    user_orders_gf[["order_id", "order_number", "order_dow",
                    "order_hour_of_day", "days_since_prior_order"]],
    on="order_id", how="left"
)
items_gf["is_flagged"] = items_gf["is_gi_sensitivity_gluten_free"].fillna(False)

# ── Summary stats ─────────────────────────────────────────────────────────────
n_orders_gf      = len(user_orders_gf)
n_items_gf       = len(items_gf)
n_flagged_gf     = items_gf["is_flagged"].sum()
pct_flagged_gf   = 100 * n_flagged_gf / n_items_gf
n_unique_gf      = items_gf["product_name"].nunique()
n_flagged_uniq_gf = items_gf[items_gf["is_flagged"]]["product_name"].nunique()
avg_items_gf     = n_items_gf / n_orders_gf
avg_days_gf      = user_orders_gf["days_since_prior_order"].dropna().mean()

print("=" * 65)
print(f"  SURVEY SNAPSHOT — Anonymous User")
print("=" * 65)
print(f"  Total orders recorded          : {n_orders_gf}")
print(f"  Total items purchased          : {n_items_gf:,}")
print(f"  Avg items per order            : {avg_items_gf:.1f}")
print(f"  Avg days between orders        : {avg_days_gf:.1f}")
print(f"  Gluten-free/GI items           : {int(n_flagged_gf):,} / {n_items_gf:,}  ({pct_flagged_gf:.1f}%)")
print(f"  Unique products ever bought    : {n_unique_gf}")
print(f"  Unique gluten-free products    : {n_flagged_uniq_gf}")

# ── Top GF products ───────────────────────────────────────────────────────────
top_gf = (items_gf[items_gf["is_flagged"]]
          .groupby("product_name").size()
          .sort_values(ascending=False)
          .head(15))

print(f"\n  Most frequently purchased gluten-free/GI items:")
for i, (name, cnt) in enumerate(top_gf.items(), 1):
    bar = "█" * min(cnt, 40)
    print(f"  {i:2d}. {name:<48}  {cnt:3d}x  {bar}")

# ── Per-order timeline (last 15 orders) ───────────────────────────────────────
print(f"\n  Order-by-order timeline (most recent 15 of {n_orders_gf}):")
print(f"  {'#':>3}  {'Day':<10}  {'Time':>5}  {'Items':>5}  {'% GF':>6}  Items purchased")
print("  " + "-" * 85)

for _, row in user_orders_gf.tail(15).iterrows():
    oid    = row["order_id"]
    onum   = int(row["order_number"])
    dow    = DOW[int(row["order_dow"])]
    time_s = f"{int(row['order_hour_of_day']):02d}:00"

    order_items_gf = items_gf[items_gf["order_id"] == oid]
    n_oi_gf        = len(order_items_gf)
    n_oi_flag_gf   = int(order_items_gf["is_flagged"].sum())
    pct_oi_gf      = 100 * n_oi_flag_gf / n_oi_gf if n_oi_gf > 0 else 0

    def fmt_gf(r):
        mark = "*" if r["is_flagged"] else " "
        return f"{mark}{r['product_name']}"
    sample_gf = ", ".join(order_items_gf.apply(fmt_gf, axis=1).tolist()[:4])
    if n_oi_gf > 4:
        sample_gf += f", … (+{n_oi_gf-4} more)"

    print(f"  {onum:3d}  {dow:<10}  {time_s:>5}  {n_oi_gf:5d}  {pct_oi_gf:5.0f}%  {sample_gf}")

print(f"\n  * = gluten-free/GI-sensitive item")
print(f"\n  What an algorithm infers from this basket history:")
print(f"    - Gluten-free substitutes across {n_flagged_uniq_gf} distinct product types over {n_orders_gf} orders")
print(f"    - Covers bread, pasta, pizza, crackers, bagels, muffins, cereal — a full GF household diet")
print(f"    - Pattern is stable and persistent over time (lift = 7.1× above chance)")
print(f"    - Consistent gluten-free purchasing is strongly predictive of celiac disease")
print(f"      or non-celiac gluten sensitivity / IBS, a medically sensitive condition")


  SURVEY SNAPSHOT — Anonymous User
  Total orders recorded          : 100
  Total items purchased          : 718
  Avg items per order            : 7.2
  Avg days between orders        : 2.3
  Gluten-free/GI items           : 174 / 718  (24.2%)
  Unique products ever bought    : 176
  Unique gluten-free products    : 31

  Most frequently purchased gluten-free/GI items:
   1. Original Gluten Free Crackers                      27x  ███████████████████████████
   2. Gluten Free Strawberry Toaster Pastry              22x  ██████████████████████
   3. Gluten Free Multigrain Bread                       16x  ████████████████
   4. Gluten Free Original English Muffins               12x  ████████████
   5. Gluten Free Rice Pasta & Cheddar Single Serving Macaroni & Cheese   12x  ████████████
   6. Gluten Free Smoked Gouda Mac and Cheese            11x  ███████████
   7. Gluten Free Whole Grain Bagels                     10x  ██████████
   8. Gluten Free Oatmeal Grahams Snack Packs 4 Ct        8